# Soccer Lineup Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.13


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.


## Problem Statement

In book problem `2.13`, the coach of Atlético Andes must assign `10` outfield players to three
positions: defense, midfield and forward. The model maximizes the total shooting score subject to:

- at least `4` defenders, `3` midfielders and `3` forwards,
- average ball skill and average speed of at least `2`,
- mutual-exclusion and implication constraints among a few players,
- forbidden `(player, position)` pairs.

The notebooks below solve the exact model printed in the book.


In [ ]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

try:
    from amplpy import AMPL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install amplpy in the active environment before running this notebook.") from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


PLAYERS = list(range(1, 13))
POSITIONS = [1, 2, 3]
HB = {1: 1, 2: 3, 3: 3, 4: 4, 5: 2, 6: 4, 7: 3, 8: 4, 9: 2, 10: 2, 11: 3, 12: 2}
SHOT = {1: 2, 2: 3, 3: 3, 4: 4, 5: 1, 6: 2, 7: 4, 8: 2, 9: 3, 10: 3, 11: 4, 12: 1}
SPEED = {1: 4, 2: 2, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 3, 9: 4, 10: 4, 11: 3, 12: 2}
DEFENSE_SKILL = {1: 2, 2: 2, 3: 3, 4: 1, 5: 4, 6: 2, 7: 1, 8: 2, 9: 1, 10: 1, 11: 4, 12: 4}
MIN_PLAY = {1: 4, 2: 3, 3: 3}
FORBIDDEN = {(1, 3), (2, 1), (4, 1), (5, 3), (6, 3), (7, 1), (7, 2), (9, 1), (11, 1), (11, 3), (12, 1)}


@timed
def solve_soccer_with_ampl(solver: str = "highs"):
    ampl = AMPL()
    ampl.eval(f"option solver {solver};")
    ampl.eval(
        r'''
        set I ordered;
        set J ordered;
        set A within {I, J};
        param HB {I};
        param D {I};
        param V {I};
        param HD {I};
        param MinPlay {J};
        var x {I, J} binary;

        maximize TotalShot:
            sum {i in I, j in J} D[i] * x[i, j];

        subject to MinPlayers {j in J}:
            sum {i in I} x[i, j] >= MinPlay[j];

        subject to MinBallSkill:
            sum {i in I, j in J} HB[i] * x[i, j] >= 20;

        subject to MinSpeed:
            sum {i in I, j in J} V[i] * x[i, j] >= 20;

        subject to Player5vs11:
            sum {j in J} x[5, j] = 1 - sum {j in J} x[11, j];

        subject to Player1vs6:
            sum {j in J} x[1, j] = sum {j in J} x[6, j];

        subject to Player1vs9:
            sum {j in J} x[1, j] = sum {j in J} x[9, j];

        subject to Player2vs4:
            sum {j in J} x[2, j] = 1 - sum {j in J} x[4, j];

        subject to TenSelected:
            sum {i in I, j in J} x[i, j] = 10;

        subject to OnePositionEach {i in I}:
            sum {j in J} x[i, j] <= 1;

        subject to ForbiddenAssignments {(i, j) in A}:
            x[i, j] = 0;
        '''
    )
    ampl.set["I"] = PLAYERS
    ampl.set["J"] = POSITIONS
    ampl.set["A"] = sorted(FORBIDDEN)
    ampl.param["HB"] = HB
    ampl.param["D"] = SHOT
    ampl.param["V"] = SPEED
    ampl.param["HD"] = DEFENSE_SKILL
    ampl.param["MinPlay"] = MIN_PLAY
    ampl.solve(solver=solver)
    values = ampl.get_variable("x").get_values().to_dict()
    defenders = sorted(player for player in PLAYERS if round(values[(player, 1)]) == 1)
    midfielders = sorted(player for player in PLAYERS if round(values[(player, 2)]) == 1)
    forwards = sorted(player for player in PLAYERS if round(values[(player, 3)]) == 1)
    selected = set(defenders) | set(midfielders) | set(forwards)
    return {
        "defenders": defenders,
        "midfielders": midfielders,
        "forwards": forwards,
        "bench": sorted(set(PLAYERS) - selected),
        "objective": int(round(ampl.get_value("TotalShot"))),
    }


## Book Model In AMPL


In [ ]:
result, elapsed = solve_soccer_with_ampl()
print("AMPL result:", result)
print(f"Elapsed time: {elapsed:.6f} seconds")
assert result["objective"] == 28
assert len(result["bench"]) == 2
